In [ ]:


import os
import kaggle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# --- 1. VLASTNÝ DATASET ---
class WikiArtDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Získa všetky názvy priečinkov (tried) a zoradí ich
        self.classes = [d for d in sorted(os.listdir(root_dir)) if os.path.isdir(os.path.join(root_dir, d))]
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.image_paths = []
        self.labels = []
        
        # Prechádzanie priečinkov a mapovanie ciest k obrázkom
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            for img_name in os.listdir(cls_dir):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(cls_dir, img_name))
                    self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# --- 2. BASELINE ARCHITEKTÚRA ---
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        # Pri vstupe 128x128 bude mať výstup po dvoch MaxPool vrstvách rozmer 32x32
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# --- VIZUALIZÁCIA ---
def visualize_batch(dataloader, classes):
    images, labels = next(iter(dataloader))
    plt.figure(figsize=(12, 6))
    for i in range(min(4, len(images))):
        plt.subplot(1, 4, i+1)
        img = images[i].permute(1, 2, 0).numpy()
        img = img * 0.5 + 0.5  # Denormalizácia pre korektné zobrazenie
        plt.imshow(img.clip(0, 1))
        # Tu je aplikovaná oprava pomocou .item()
        plt.title(classes[labels[i].item()][:15] + '...') 
        plt.axis('off')
    plt.tight_layout()
    plt.show()

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Používam zariadenie: {device}")

    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    data_dir = "wikiart/versions/1"
    dataset = WikiArtDataset(root_dir=data_dir, transform=transform)
    print(f"Dataset načítaný. Celkový počet obrázkov: {len(dataset)}, Počet tried: {len(dataset.classes)}")
    
    # --- 3. ROZDELENIE DÁT A DATALOADERY ---
    val_size = int(0.2 * len(dataset))
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    
    # Vizualizácia jedného batchu po transformácii[cite: 1]
    visualize_batch(train_loader, dataset.classes)

    model = SimpleCNN(num_classes=len(dataset.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # --- 4. EXPLICITNÁ TRÉNOVACIA A VALIDAČNÁ SLUČKA ---
    epochs = 10
    for epoch in range(epochs):
        # -- TRÉNOVACIA FÁZA --
        model.train() # Nastavenie modelu do trénovacieho režimu[cite: 1]
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad() # Vynulovanie gradientov[cite: 1]
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward() # Spätný prechod a výpočet gradientov[cite: 1]
            optimizer.step() # Aktualizácia váh[cite: 1]
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
        train_acc = 100 * train_correct / train_total
        
        # -- VALIDAČNÁ FÁZA --
        model.eval() # Nastavenie modelu do vyhodnocovacieho režimu[cite: 1]
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad(): # Zabránenie úniku pamäte vo validačnej slučke[cite: 1]
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss:   {val_loss/len(val_loader):.4f} | Val Acc:   {val_acc:.2f}%")

if __name__ == "__main__":
    main()

: 